# What the screen can and cannot resolve



In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
TABLES = Path("../results/tables")
FIGURES = Path("../results/figures")
TABLES.mkdir(parents=True, exist_ok=True)

from perturbseq.estimators import lane_counts
from perturbseq.estimators import mh_log_or

*from `08_representation.py`*

08 - Sensitivity to perturbation representation.

Writes
    results/tables/08_downsample_curve.tsv      effect and detection vs cells per element
    results/tables/08_detectable_effect.tsv     smallest detectable effect at each cell count
    results/tables/08_underpowered_elements.tsv elements where the screen cannot answer

This is deliberately *not* a formal power analysis. There is one infection
experiment here, so there is no replicate structure to estimate a between-
experiment variance component from, and any "power" number would be a
statement about cell-level sampling noise dressed up as a statement about the
experiment. What can be said honestly is narrower and still useful:

  - how the estimated effect and its uncertainty behave as an element's cell
    count is reduced, holding the biology fixed (subsampling real cells);
  - what effect size the NTC-based null can separate from zero at a given cell
    count, which is a property of the screen's coverage;
  - which library elements have too few cells for either phenotype, so that
    their absence from the hit list is not read as evidence of no effect.

The last point is the one that matters for interpreting a screen: 47 of 237
elements have fewer than 55 cells, and the paper drops 48 for the same reason.
Those elements are mostly essential genes, i.e. the ones whose knockdown kills
cells, so their under-representation is informative rather than random.

In [2]:
DRAWS = 400

SUBSAMPLE_SIZES = [25, 40, 60, 80, 120, 160]

RNG = np.random.default_rng(0)

d = pd.read_csv(TABLES / "03_cell_state.tsv.gz", sep="\t")
d["gem_group"] = d.gem_group.astype(str)
lanes = sorted(d.gem_group.unique())
ntc = d[d.is_ntc]
c_l, d_l = lane_counts(ntc, lanes, "state")
p_l = np.divide(c_l, c_l + d_l, out=np.zeros(len(lanes)), where=(c_l + d_l) > 0)

# Pick real hits spanning the whole effect-size range, not just the biggest
# ones: a subsampling curve built only on ACE2-sized effects says nothing
# about the elements whose detection is actually marginal.
hits = pd.read_csv(TABLES / "07_hits_infection.tsv", sep="\t")
sig = hits[(~hits.is_ntc) & (hits.q_null < 0.1)].copy()
sig["abs_eff"] = sig.log_or_mh.abs()
sig = sig.sort_values("abs_eff")
picks = np.unique(np.linspace(0, len(sig) - 1, 8).round().astype(int))
strong = sig.iloc[picks]
print("elements used for the subsampling curve (weakest to strongest):")
print(strong[["target", "n_cells", "log_or_mh", "q_null"]].round(3).to_string(index=False))

elements used for the subsampling curve (weakest to strongest):
      target  n_cells  log_or_mh  q_null
    TMEM106B      380     -0.317   0.047
       SPNS1      177     -0.490   0.039
       SMAD3      107      0.538   0.045
      ARFRP1      155     -0.611   0.006
       SEC62       84      0.788   0.036
     TMPRSS2      184     -1.212   0.006
TMPRSS2+CTSL      205     -1.462   0.006
TMPRSS2+CTSB      174     -1.808   0.006


In [3]:
# ---------------------------------------- subsample real cells of real hits
rows = []
for _, h in strong.iterrows():
    sub = d[d.element == h.element]
    for k in SUBSAMPLE_SIZES + [len(sub)]:
        if k > len(sub):
            continue
        eff, hit = [], 0
        for _ in range(60):
            take = sub.sample(k, random_state=int(RNG.integers(1e9)))
            a_l, b_l = lane_counts(take, lanes, "state")
            lo, _se = mh_log_or(a_l, b_l, c_l, d_l)
            if not np.isfinite(lo):
                continue
            n_l = a_l + b_l
            a_null = RNG.binomial(np.repeat(n_l[None, :], DRAWS, axis=0), p_l[None, :])
            b_null = n_l[None, :] - a_null
            with np.errstate(divide="ignore", invalid="ignore"):
                null = np.log(
                    np.sum(a_null * d_l / (n_l + c_l + d_l), axis=1)
                    / np.sum(b_null * c_l / (n_l + c_l + d_l), axis=1)
                )
            p = (1 + np.sum(np.abs(null[np.isfinite(null)]) >= abs(lo))) / (DRAWS + 1)
            eff.append(lo)
            hit += int(p < 0.05)
        if eff:
            rows.append(
                dict(element=h.element, target=h.target, cells=k,
                     mean_log_or=float(np.mean(eff)),
                     sd_log_or=float(np.std(eff)),
                     detection_rate=hit / len(eff),
                     full_log_or=h.log_or_mh)
            )
curve = pd.DataFrame(rows)
curve.to_csv(TABLES / "08_downsample_curve.tsv", sep="\t", index=False)
print("\nsubsampling real hits: detection rate at p<0.05, by cells retained")
print(
    curve[curve.cells.isin(SUBSAMPLE_SIZES)]
    .pivot_table(index="cells", columns="target", values="detection_rate")
    .round(2).to_string()
)
print("\nand the effect estimate itself (mean log odds ratio):")
print(
    curve[curve.cells.isin(SUBSAMPLE_SIZES)]
    .pivot_table(index="cells", columns="target", values="mean_log_or")
    .round(2).to_string()
)


subsampling real hits: detection rate at p<0.05, by cells retained
target  ARFRP1  SEC62  SMAD3  SPNS1  TMEM106B  TMPRSS2  TMPRSS2+CTSB  TMPRSS2+CTSL
cells                                                                             
25        0.25   0.28   0.13   0.32      0.12     0.66          0.98          0.81
40        0.33   0.62   0.28   0.25      0.13     0.98          1.00          0.98
60        0.55   0.87   0.52   0.45      0.20     1.00          1.00          1.00
80        0.73   1.00   0.58   0.37      0.25     1.00          1.00          1.00
120       0.98    NaN    NaN   0.75      0.35     1.00          1.00          1.00
160        NaN    NaN    NaN   1.00      0.35     1.00          1.00          1.00

and the effect estimate itself (mean log odds ratio):
target  ARFRP1  SEC62  SMAD3  SPNS1  TMEM106B  TMPRSS2  TMPRSS2+CTSB  TMPRSS2+CTSL
cells                                                                             
25       -0.70   0.73   0.56  -0.61     -0.34  

In [4]:
# -------------------------------- smallest detectable effect per cell count
# The null's spread at a given cell count sets the resolution of the screen.
rows = []
for k in [25, 40, 60, 80, 120, 160, 240]:
    n_l = np.full(len(lanes), k / len(lanes))
    n_l = np.floor(n_l).astype(int)
    n_l[: k - n_l.sum()] += 1
    a_null = RNG.binomial(np.repeat(n_l[None, :], 4000, axis=0), p_l[None, :])
    b_null = n_l[None, :] - a_null
    with np.errstate(divide="ignore", invalid="ignore"):
        null = np.log(
            np.sum(a_null * d_l / (n_l + c_l + d_l), axis=1)
            / np.sum(b_null * c_l / (n_l + c_l + d_l), axis=1)
        )
    null = null[np.isfinite(null)]
    rows.append(
        dict(cells=k,
             null_sd=float(np.std(null)),
             abs_log_or_at_p05=float(np.percentile(np.abs(null), 95)),
             fold_change_at_p05=float(np.exp(np.percentile(np.abs(null), 95)))))
det = pd.DataFrame(rows)
det.to_csv(TABLES / "08_detectable_effect.tsv", sep="\t", index=False)
print("\nresolution of the screen, from the NTC null alone:")
print(det.round(3).to_string(index=False))


resolution of the screen, from the NTC null alone:
 cells  null_sd  abs_log_or_at_p05  fold_change_at_p05
    25    0.456              0.885               2.424
    40    0.354              0.652               1.919
    60    0.273              0.545               1.725
    80    0.238              0.477               1.611
   120    0.194              0.390               1.476
   160    0.167              0.327               1.387
   240    0.137              0.273               1.313


In [5]:
# ------------------------------------------------- underpowered elements
rep = pd.read_csv(TABLES / "04_element_representation.tsv", sep="\t")
rep["testable_infection"] = rep.n_cells >= 55
rep["testable_burden"] = rep.n_infected >= 20
rep["testable_isg"] = rep.n_bystander >= 30
under = rep[~(rep.testable_infection & rep.testable_burden & rep.testable_isg)]
under.to_csv(TABLES / "08_underpowered_elements.tsv", sep="\t", index=False)
print(
    f"\nelements of {len(rep)} not testable for at least one phenotype: {len(under)}"
    f"\n  infection (>=55 cells)      : {int((~rep.testable_infection).sum())} fail"
    f"\n  burden   (>=20 infected)    : {int((~rep.testable_burden).sum())} fail"
    f"\n  ISG      (>=30 bystanders)  : {int((~rep.testable_isg).sum())} fail"
)
print("\nthe 15 worst-represented elements (absence from a hit list means nothing here):")
print(
    under.nsmallest(15, "n_cells")[
        ["element", "target", "n_cells", "n_infected", "n_bystander"]
    ].to_string(index=False)
)


elements of 237 not testable for at least one phenotype: 73
  infection (>=55 cells)      : 47 fail
  burden   (>=20 infected)    : 67 fail
  ISG      (>=30 bystanders)  : 47 fail

the 15 worst-represented elements (absence from a hit list means nothing here):
                  element        target  n_cells  n_infected  n_bystander
        CRISPRi_039_COPB2         COPB2        1           0            1
       CRISPRi_093_POLR2B        POLR2B        1           1            0
         CRISPRi_061_GBF1          GBF1        4           2            0
      CRISPRi_012_ATP6V1A       ATP6V1A        5           0            2
         CRISPRi_150_YKT6          YKT6        5           2            3
        CRISPRi_119_SRP54         SRP54        6           3            3
CRISPRi_176_SREBF1_SREBF2 SREBF1+SREBF2        6           3            1
     CRISPRi_013_ATP6V1B2      ATP6V1B2        6           1            4
        CRISPRi_091_PCBP1         PCBP1        8           0            